# the project root (all-in-one)

Todo el código del baseline está en esta libreta. **Los datos** (`plagiarism_labels.csv`, carpetas de clones) deben estar en esta misma carpeta. Instala dependencias con `requirements.txt`.

**Colab:** zip this folder, upload, unzip, `%cd` into `the project root`, then **Run all**.


In [ ]:
print("[Paso dependencias] Instalando paquetes desde requirements.txt ...")
%pip install -q -r requirements.txt
print("[Paso dependencias] Listo.")


In [ ]:
from __future__ import annotations

import argparse
import io
import itertools
import json
import logging
import random
import re
import tokenize
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

import os
import sys


def resolve_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for base in candidates:
        base = base.resolve()
        if (base / "requirements.txt").is_file() and (
            base / "plagiarism_labels.csv"
        ).is_file():
            return base
    raise FileNotFoundError(
        "Could not locate project root (need requirements.txt and plagiarism_labels.csv). "
        "On Colab: unzip project, then %cd into the project folder before running."
    )


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- shared snippet segmentation ---
LANG_MARKER_RE = re.compile(
    r"(?im)^\s*(?:python|java|javascript|c\+\+|cpp|ruby|go)\s*$"
)
SNIPPET_SEP_RE = re.compile(
    r"\n(?:\s*\n){2,}(?=\s*(?:def|class|from|import|@)\b)"
)
FALLBACK_SEP_RE = re.compile(r"\n\s*\n\s*\n+")


def split_snippets(file_text: str) -> list[str]:
    #aplicamos la misma idea de segmentacion usada para generar pares originalmente
    text = file_text.replace("\r\n", "\n").replace("\r", "\n").strip()
    if not text:
        return []

    #separamos bloques con marcadores de lenguaje cuando aparecen en el archivo
    blocks = [block.strip() for block in LANG_MARKER_RE.split(text) if block.strip()]
    snippets: list[str] = []

    #intentamos separar por bloques con saltos amplios antes de def class import
    for block in blocks:
        parts = [part.strip() for part in SNIPPET_SEP_RE.split(block) if part.strip()]
        snippets.extend(parts if len(parts) > 1 else [block])

    #usamos un respaldo por triples saltos de linea si no logramos al menos dos snippets
    if len(snippets) < 2:
        fallback_parts = [part.strip() for part in FALLBACK_SEP_RE.split(text) if part.strip()]
        if len(fallback_parts) > len(snippets):
            snippets = fallback_parts

    return snippets

# --- baseline.config ---
# aqui centralizamos parametros del baseline para mantener una configuracion clara y reproducible
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class BaselineConfig:
    # aqui guardamos rutas y parametros globales que se usan en todo el pipeline
    dataset_root: Path
    metadata_filename: str = "clone_pairs_dataset_metadata.csv"
    seed: int = 42
    train_size: float = 0.7
    val_size: float = 0.15
    test_size: float = 0.15
    imbalance_threshold: float = 1.5

    @property
    def metadata_path(self) -> Path:
        # aqui construimos la ruta completa al csv metadata de entrada
        return self.dataset_root / self.metadata_filename

# --- baseline.utils ---
#reunimos utilidades simples para crear carpetas, logs y archivos de salida del baseline
import json
import logging
import random
from pathlib import Path
from typing import Any

import numpy as np


def ensure_dir(path: Path) -> Path:
    #nos aseguramos de que la carpeta exista antes de guardar resultados
    path.mkdir(parents=True, exist_ok=True)
    return path


def setup_logging(log_path: Path) -> logging.Logger:
    #preparamos un logger con salida a archivo y a consola para trazabilidad
    ensure_dir(log_path.parent)
    logger = logging.getLogger("clone_baseline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    #definimos un formato legible para ver fecha, nivel y mensaje
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    #conectamos el archivo de log principal de la corrida
    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    #conectamos tambien consola para seguimiento en tiempo real
    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)
    logger.addHandler(stream_handler)

    return logger


def set_global_seed(seed: int) -> None:
    #fijamos semillas para que los resultados sean reproducibles
    random.seed(seed)
    np.random.seed(seed)


def save_json(payload: dict[str, Any], output_path: Path) -> None:
    #guardamos diccionarios en json de forma consistente
    ensure_dir(output_path.parent)
    with output_path.open("w", encoding="utf-8") as fp:
        json.dump(payload, fp, indent=2, ensure_ascii=False)


def class_distribution(series) -> dict[str, int]:
    #calculamos distribucion de clases para reportes y control de desbalance
    counts = series.value_counts(dropna=False)
    return {str(k): int(v) for k, v in counts.items()}


def imbalance_ratio(series) -> float:
    #medimos que tan desbalanceada esta una variable objetivo
    counts = series.value_counts()
    if counts.empty:
        return 1.0
    if len(counts) == 1:
        return float("inf")
    return float(counts.max() / counts.min())

# --- baseline.preprocessing ---
#hacemos limpieza ligera y tokenizacion python sin romper la logica del codigo
import io
import re
import tokenize
from typing import Iterable


#declaramos reglas de normalizacion de espacios y lineas vacias
WHITESPACE_RE = re.compile(r"[ \t]+")
MULTI_NEWLINE_RE = re.compile(r"\n{3,}")


def strip_comments(code: str) -> str:
    #quitamos comentarios para reducir ruido lexical en las features
    if not code.strip():
        return code

    try:
        output_tokens: list[tokenize.TokenInfo] = []
        reader = io.StringIO(code).readline
        for tok in tokenize.generate_tokens(reader):
            if tok.type == tokenize.COMMENT:
                continue
            output_tokens.append(tok)
        return tokenize.untokenize(output_tokens)
    except (tokenize.TokenError, IndentationError):
        #dejamos el codigo original si tokenizar falla en snippets ruidosos
        return code


def normalize_whitespace(code: str) -> str:
    #compactamos espacios repetidos y limitamos saltos de linea consecutivos
    lines = []
    for line in code.splitlines():
        compact = WHITESPACE_RE.sub(" ", line).rstrip()
        lines.append(compact)
    normalized = "\n".join(lines).strip()
    normalized = MULTI_NEWLINE_RE.sub("\n\n", normalized)
    return normalized


def preprocess_code(code: str) -> str:
    #aplicamos el preprocesamiento minimo que acordamos para el baseline
    no_comments = strip_comments(code)
    return normalize_whitespace(no_comments)


def tokenize_python_code(code: str) -> list[str]:
    #tokenizamos con la libreria tokenize para respetar mejor sintaxis python
    if not code.strip():
        return []

    try:
        tokens: list[str] = []
        reader = io.StringIO(code).readline
        skip_types = {
            tokenize.ENCODING,
            tokenize.ENDMARKER,
            tokenize.NL,
            tokenize.NEWLINE,
            tokenize.INDENT,
            tokenize.DEDENT,
            tokenize.COMMENT,
        }
        for tok in tokenize.generate_tokens(reader):
            if tok.type in skip_types:
                continue
            text = tok.string.strip()
            if text:
                tokens.append(text)
        return tokens
    except (tokenize.TokenError, IndentationError):
        #usamos un respaldo regex cuando el snippet tiene formato incompleto
        return re.findall(r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|[-+*/%=<>()[\]{},.:;]", code)


def jaccard_similarity(tokens_a: Iterable[str], tokens_b: Iterable[str]) -> float:
    #medimos interseccion sobre union de tokens unicos
    set_a = set(tokens_a)
    set_b = set(tokens_b)
    if not set_a and not set_b:
        return 1.0
    union = set_a | set_b
    if not union:
        return 0.0
    return float(len(set_a & set_b) / len(union))


def overlap_ratio(tokens_a: Iterable[str], tokens_b: Iterable[str]) -> float:
    #medimos que tanto se cubre el conjunto mas pequeno de tokens
    set_a = set(tokens_a)
    set_b = set(tokens_b)
    if not set_a and not set_b:
        return 1.0
    min_size = min(len(set_a), len(set_b))
    if min_size == 0:
        return 0.0
    return float(len(set_a & set_b) / min_size)


def dice_similarity(tokens_a: Iterable[str], tokens_b: Iterable[str]) -> float:
    #calculamos el coeficiente de Dice para mantener el mismo patrón de similitud de clase
    set_a = set(tokens_a)
    set_b = set(tokens_b)
    if not set_a and not set_b:
        return 1.0
    denom = len(set_a) + len(set_b)
    if denom == 0:
        return 0.0
    return float(2.0 * len(set_a & set_b) / denom)

# --- baseline.evaluate_baseline ---
#calculamos metricas reportes y graficas para evaluar cada modelo del baseline
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    RocCurveDisplay,
)


def evaluate_predictions(
    y_true,
    y_pred,
    labels: list,
    y_proba=None,
) -> dict[str, Any]:
    #calculamos metricas globales y reportes detallados por clase
    acc = accuracy_score(y_true, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    report_dict = classification_report(
        y_true, y_pred, labels=labels, output_dict=True, zero_division=0
    )
    report_text = classification_report(y_true, y_pred, labels=labels, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    per_class_precision, per_class_recall, per_class_f1, per_class_support = (
        precision_recall_fscore_support(
            y_true, y_pred, labels=labels, average=None, zero_division=0
        )
    )

    #alineado con prácticas de clase: métricas por clase y AUC/ROC cuando hay probabilidades
    #seguimos la práctica de clase y guardamos métricas por clase en forma explícita
    per_class_metrics = {
        str(label): {
            "precision": float(per_class_precision[idx]),
            "recall": float(per_class_recall[idx]),
            "f1": float(per_class_f1[idx]),
            "support": int(per_class_support[idx]),
        }
        for idx, label in enumerate(labels)
    }

    roc_auc = None
    if y_proba is not None:
        #cuando tenemos probabilidades, también calculamos AUC para complementar F1 y accuracy
        try:
            proba_arr = np.asarray(y_proba)
            if len(labels) == 2:
                #en binario usamos la probabilidad de la clase positiva
                roc_auc = float(roc_auc_score(y_true, proba_arr[:, 1]))
            else:
                #en multiclase usamos esquema one-vs-rest para mantener una sola métrica global
                roc_auc = float(
                    roc_auc_score(y_true, proba_arr, labels=labels, multi_class="ovr")
                )
        except (ValueError, IndexError):
            roc_auc = None

    return {
        "accuracy": float(acc),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
        "confusion_matrix": cm.tolist(),
        "classification_report_dict": report_dict,
        "classification_report_text": report_text,
        "per_class_metrics": per_class_metrics,
        "roc_auc": roc_auc,
    }


def save_roc_curve_plot(
    y_true,
    y_score: np.ndarray,
    output_path: Path,
    title: str,
    pos_label=1,
) -> None:
    #graficamos curva roc al estilo de clase cuando tenemos tarea binaria
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_predictions(y_true, y_score, pos_label=pos_label, ax=ax)
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def save_confusion_matrix_plot(
    confusion: np.ndarray,
    labels: list[str],
    title: str,
    output_path: Path,
) -> None:
    #dibujamos y guardamos la matriz de confusion para cada modelo y split
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots(figsize=(6, 5))
    image = ax.imshow(confusion, interpolation="nearest", cmap=plt.cm.Blues)
    ax.figure.colorbar(image, ax=ax)
    ax.set(
        xticks=np.arange(len(labels)),
        yticks=np.arange(len(labels)),
        xticklabels=labels,
        yticklabels=labels,
        ylabel="True label",
        xlabel="Predicted label",
        title=title,
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    #agregamos valores dentro de cada celda para lectura rapida
    threshold = confusion.max() / 2.0 if confusion.size else 0.0
    for i in range(confusion.shape[0]):
        for j in range(confusion.shape[1]):
            ax.text(
                j,
                i,
                format(confusion[i, j], "d"),
                ha="center",
                va="center",
                color="white" if confusion[i, j] > threshold else "black",
            )

    fig.tight_layout()
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def comparison_table(results: list[dict[str, Any]]) -> pd.DataFrame:
    #armamos una tabla plana para comparar modelos entre tareas y splits
    rows = []
    for result in results:
        rows.append(
            {
                "task": result["task"],
                "model": result["model_name"],
                "split": result["split"],
                "accuracy": result["metrics"]["accuracy"],
                "precision_macro": result["metrics"]["precision_macro"],
                "recall_macro": result["metrics"]["recall_macro"],
                "f1_macro": result["metrics"]["f1_macro"],
                "precision_weighted": result["metrics"]["precision_weighted"],
                "recall_weighted": result["metrics"]["recall_weighted"],
                "f1_weighted": result["metrics"]["f1_weighted"],
            }
        )
    return pd.DataFrame(rows)

# --- baseline.data_loading ---
#concentramos carga validacion limpieza y split por grupos para evitar leakage
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit


#declaramos las columnas que esperamos en el metadata
REQUIRED_METADATA_COLUMNS = [
    "is_clone",
    "clone_type",
    "source_group",
    "filename",
    "file_path",
    "problem_id",
    "snippet_index_a",
    "snippet_index_b",
]


@dataclass
class SplitResult:
    #guardamos los indices de train val y test ya separados por grupos
    train_idx: pd.Index
    val_idx: pd.Index
    test_idx: pd.Index


def load_metadata_csv(path: Path) -> pd.DataFrame:
    #cargamos el csv metadata y fallamos temprano si no existe
    if not path.exists():
        raise FileNotFoundError(f"Metadata CSV not found: {path}")
    return pd.read_csv(path)


def validate_metadata_schema(metadata_df: pd.DataFrame) -> tuple[bool, list[str]]:
    #revisamos que el metadata tenga columnas y valores basicos correctos
    errors: list[str] = []

    missing_cols = [col for col in REQUIRED_METADATA_COLUMNS if col not in metadata_df.columns]
    if missing_cols:
        errors.append(f"Missing required columns: {missing_cols}")

    if "filename" in metadata_df.columns and metadata_df["filename"].isna().any():
        errors.append("Column `filename` has null values.")
    if "file_path" in metadata_df.columns and metadata_df["file_path"].isna().any():
        errors.append("Column `file_path` has null values.")
    if "problem_id" in metadata_df.columns and metadata_df["problem_id"].isna().any():
        errors.append("Column `problem_id` has null values.")
    if "is_clone" in metadata_df.columns:
        allowed = {0, 1}
        found = set(pd.to_numeric(metadata_df["is_clone"], errors="coerce").dropna().unique())
        if not found.issubset(allowed):
            errors.append(f"Column `is_clone` has invalid values: {sorted(found - allowed)}")

    return len(errors) == 0, errors


def clean_metadata_rows(metadata_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    #convertimos tipos y apartamos filas con valores faltantes o invalidos
    cleaned = metadata_df.copy()

    cleaned["is_clone"] = pd.to_numeric(cleaned["is_clone"], errors="coerce")
    cleaned["problem_id"] = pd.to_numeric(cleaned["problem_id"], errors="coerce")
    cleaned["snippet_index_a"] = pd.to_numeric(cleaned["snippet_index_a"], errors="coerce")
    cleaned["snippet_index_b"] = pd.to_numeric(cleaned["snippet_index_b"], errors="coerce")

    invalid_mask = (
        cleaned["is_clone"].isna()
        | cleaned["problem_id"].isna()
        | cleaned["snippet_index_a"].isna()
        | cleaned["snippet_index_b"].isna()
        | cleaned["file_path"].isna()
        | cleaned["filename"].isna()
    )

    invalid_rows = cleaned.loc[invalid_mask].copy()
    if not invalid_rows.empty:
        invalid_rows["drop_reason"] = "invalid_or_missing_metadata_values"

    #nos quedamos solo con filas utilizables y fijamos tipos enteros
    cleaned = cleaned.loc[~invalid_mask].copy()
    cleaned["is_clone"] = cleaned["is_clone"].astype(int)
    cleaned["problem_id"] = cleaned["problem_id"].astype(int)
    cleaned["snippet_index_a"] = cleaned["snippet_index_a"].astype(int)
    cleaned["snippet_index_b"] = cleaned["snippet_index_b"].astype(int)

    return cleaned, invalid_rows


def _distribution_distance(
    full_df: pd.DataFrame,
    splits: dict[str, pd.DataFrame],
    target_col: str,
) -> float:
    #medimos que tan parecida queda la distribucion de clases entre splits y dataset completo
    overall = full_df[target_col].value_counts(normalize=True)
    classes = overall.index.tolist()
    distance = 0.0
    for split_df in splits.values():
        split_dist = split_df[target_col].value_counts(normalize=True)
        split_dist = split_dist.reindex(classes, fill_value=0.0)
        distance += float((split_dist - overall.reindex(classes, fill_value=0.0)).abs().sum())
    return distance


def grouped_train_val_test_split(
    df: pd.DataFrame,
    group_col: str,
    target_col: str,
    seed: int = 42,
    train_size: float = 0.7,
    val_size: float = 0.15,
    test_size: float = 0.15,
    max_attempts: int = 100,
) -> SplitResult:
    #hacemos split por grupos para que un mismo problem_id no caiga en train y test
    if round(train_size + val_size + test_size, 8) != 1.0:
        raise ValueError("train_size + val_size + test_size must sum to 1.0")

    best: tuple[float, SplitResult] | None = None
    required_classes = set(df[target_col].unique())
    temp_ratio = val_size + test_size
    rel_test_size = test_size / temp_ratio

    #intentamos varias semillas para mantener clases presentes y mejor equilibrio
    for attempt in range(max_attempts):
        random_seed = seed + attempt
        gss_train = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_seed)
        train_idx_arr, temp_idx_arr = next(
            gss_train.split(df, y=df[target_col], groups=df[group_col])
        )

        temp_df = df.iloc[temp_idx_arr]
        gss_temp = GroupShuffleSplit(
            n_splits=1, test_size=rel_test_size, random_state=random_seed
        )
        val_rel_idx, test_rel_idx = next(
            gss_temp.split(temp_df, y=temp_df[target_col], groups=temp_df[group_col])
        )

        train_idx = df.index[train_idx_arr]
        val_idx = temp_df.index[val_rel_idx]
        test_idx = temp_df.index[test_rel_idx]

        train_df = df.loc[train_idx]
        val_df = df.loc[val_idx]
        test_df = df.loc[test_idx]

        #pedimos que todas las clases existan en cada split cuando sea posible
        if required_classes.issubset(set(train_df[target_col].unique())) and required_classes.issubset(
            set(val_df[target_col].unique())
        ) and required_classes.issubset(set(test_df[target_col].unique())):
            distance = _distribution_distance(
                full_df=df,
                splits={"train": train_df, "val": val_df, "test": test_df},
                target_col=target_col,
            )
            result = SplitResult(train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)
            if best is None or distance < best[0]:
                best = (distance, result)

    if best is None:
        #dejamos un respaldo si no encontramos una particion perfecta en los intentos
        gss_train = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
        train_idx_arr, temp_idx_arr = next(
            gss_train.split(df, y=df[target_col], groups=df[group_col])
        )
        temp_df = df.iloc[temp_idx_arr]
        gss_temp = GroupShuffleSplit(n_splits=1, test_size=rel_test_size, random_state=seed)
        val_rel_idx, test_rel_idx = next(
            gss_temp.split(temp_df, y=temp_df[target_col], groups=temp_df[group_col])
        )
        return SplitResult(
            train_idx=df.index[train_idx_arr],
            val_idx=temp_df.index[val_rel_idx],
            test_idx=temp_df.index[test_rel_idx],
        )

    return best[1]


def split_statistics(
    df: pd.DataFrame,
    split_col: str,
    target_col: str,
    group_col: str,
) -> list[dict[str, Any]]:
    #resumimos filas grupos y distribucion de clases por split para auditar el corte
    stats: list[dict[str, Any]] = []
    for split_name, split_df in df.groupby(split_col):
        counts = split_df[target_col].value_counts().to_dict()
        stats.append(
            {
                "split": split_name,
                "rows": int(len(split_df)),
                "unique_groups": int(split_df[group_col].nunique()),
                "class_distribution": {str(k): int(v) for k, v in counts.items()},
            }
        )
    return stats


def balance_training_dataframe(
    train_df: pd.DataFrame,
    target_col: str,
    strategy: str = "undersample",
    seed: int = 42,
) -> tuple[pd.DataFrame, dict[str, Any]]:
    #balanceamos solo el split de entrenamiento para reducir sesgo por clases mayoritarias
    if strategy not in {"none", "undersample", "oversample"}:
        raise ValueError("strategy must be one of: none, undersample, oversample")

    original_counts = train_df[target_col].value_counts()
    info: dict[str, Any] = {
        "strategy": strategy,
        "target_col": target_col,
        "rows_before": int(len(train_df)),
        "class_distribution_before": {str(k): int(v) for k, v in original_counts.items()},
    }

    if strategy == "none" or len(original_counts) <= 1:
        info["rows_after"] = int(len(train_df))
        info["class_distribution_after"] = info["class_distribution_before"]
        return train_df.copy(), info

    if strategy == "undersample":
        target_size = int(original_counts.min())
        replace = False
    else:
        target_size = int(original_counts.max())
        replace = True

    balanced_parts: list[pd.DataFrame] = []
    for class_value in original_counts.index.tolist():
        class_df = train_df[train_df[target_col] == class_value]
        balanced_parts.append(
            class_df.sample(
                n=target_size,
                replace=replace,
                random_state=seed,
            )
        )

    balanced_df = pd.concat(balanced_parts, axis=0)
    balanced_df = balanced_df.sample(frac=1.0, random_state=seed).copy()

    balanced_counts = balanced_df[target_col].value_counts()
    info["rows_after"] = int(len(balanced_df))
    info["class_distribution_after"] = {str(k): int(v) for k, v in balanced_counts.items()}
    return balanced_df, info

# --- baseline.snippet_reconstruction ---
def normalize_relative_path(raw_path: str) -> Path:
    #normalizamos separadores para que pathlib resuelva rutas de forma estable
    return Path(str(raw_path).replace("\\", "/"))


def reconstruct_pairs_from_metadata(
    metadata_df: pd.DataFrame,
    dataset_root: Path,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    #recorremos cada fila y validamos ruta lectura y rango de indices
    records: list[dict[str, Any]] = []
    dropped: list[dict[str, Any]] = []
    snippet_cache: dict[str, list[str]] = {}
    cache_path_failures: set[str] = set()

    for row in metadata_df.itertuples(index=False):
        #resolvemos la ruta del archivo que contiene los snippets de esta fila
        raw_path = str(row.file_path)
        rel_path = normalize_relative_path(raw_path)
        abs_path = dataset_root / rel_path

        #descartamos filas que apuntan a archivos inexistentes
        if not abs_path.exists():
            dropped.append(
                {
                    **row._asdict(),
                    "drop_reason": "missing_file_path",
                    "resolved_path": str(abs_path),
                }
            )
            continue

        #cacheamos snippets por archivo para evitar trabajo repetido
        cache_key = str(rel_path).lower()
        if cache_key not in snippet_cache and cache_key not in cache_path_failures:
            try:
                file_text = abs_path.read_text(encoding="utf-8", errors="replace")
                snippet_cache[cache_key] = split_snippets(file_text)
            except OSError:
                #registramos error de lectura y seguimos con la siguiente fila
                cache_path_failures.add(cache_key)
                dropped.append(
                    {
                        **row._asdict(),
                        "drop_reason": "file_read_error",
                        "resolved_path": str(abs_path),
                    }
                )
                continue

        #verificamos que el archivo tenga snippets ya cargados en cache
        snippets = snippet_cache.get(cache_key)
        if snippets is None:
            dropped.append(
                {
                    **row._asdict(),
                    "drop_reason": "snippet_cache_failure",
                    "resolved_path": str(abs_path),
                }
            )
            continue

        #validamos indices para no usar pares invalidos
        idx_a = int(row.snippet_index_a)
        idx_b = int(row.snippet_index_b)
        if idx_a == idx_b:
            dropped.append(
                {
                    **row._asdict(),
                    "drop_reason": "identical_snippet_indices",
                    "resolved_path": str(abs_path),
                    "snippet_count": len(snippets),
                }
            )
            continue
        if idx_a < 0 or idx_b < 0:
            dropped.append(
                {
                    **row._asdict(),
                    "drop_reason": "negative_snippet_index",
                    "resolved_path": str(abs_path),
                    "snippet_count": len(snippets),
                }
            )
            continue
        if idx_a >= len(snippets) or idx_b >= len(snippets):
            dropped.append(
                {
                    **row._asdict(),
                    "drop_reason": "snippet_index_out_of_range",
                    "resolved_path": str(abs_path),
                    "snippet_count": len(snippets),
                }
            )
            continue

        #guardamos la fila reconstruida con snippets y metadatos
        records.append(
            {
                **row._asdict(),
                "resolved_path": str(abs_path),
                "snippet_count": len(snippets),
                "code_a": snippets[idx_a],
                "code_b": snippets[idx_b],
            }
        )

    #dejamos dos dataframes para auditoria de calidad
    reconstructed_df = pd.DataFrame(records)
    dropped_df = pd.DataFrame(dropped)

    #resumimos conteos para reporte
    summary = {
        "metadata_rows": int(len(metadata_df)),
        "reconstructed_rows": int(len(reconstructed_df)),
        "dropped_rows": int(len(dropped_df)),
        "drop_reasons": (
            dropped_df["drop_reason"].value_counts().to_dict() if not dropped_df.empty else {}
        ),
    }
    return reconstructed_df, dropped_df, summary


# --- baseline.feature_extraction ---
#construimos representaciones de texto y features numericas del par de snippets
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer


def prepare_pair_text_fields(df: pd.DataFrame) -> pd.DataFrame:
    #limpiamos cada snippet y generamos tokens para ambos lados del par
    result = df.copy()

    code_a_clean = [preprocess_code(text) for text in result["code_a"].astype(str)]
    code_b_clean = [preprocess_code(text) for text in result["code_b"].astype(str)]

    tokens_a = [tokenize_python_code(text) for text in code_a_clean]
    tokens_b = [tokenize_python_code(text) for text in code_b_clean]

    #guardamos campos intermedios para trazabilidad y para calcular features
    result["code_a_clean"] = code_a_clean
    result["code_b_clean"] = code_b_clean
    result["tokens_a"] = tokens_a
    result["tokens_b"] = tokens_b
    result["token_text_a"] = [" ".join(tokens) for tokens in tokens_a]
    result["token_text_b"] = [" ".join(tokens) for tokens in tokens_b]
    return result


def fit_tfidf_vectorizer(train_df: pd.DataFrame) -> TfidfVectorizer:
    #ajustamos tf idf solo con train para evitar leakage hacia val y test
    corpus = pd.concat([train_df["token_text_a"], train_df["token_text_b"]], axis=0)
    vectorizer = TfidfVectorizer(
        tokenizer=str.split,
        preprocessor=None,
        token_pattern=None,
        lowercase=False,
        ngram_range=(1, 2),
        min_df=1,
    )
    vectorizer.fit(corpus)
    return vectorizer


def rowwise_cosine_similarity(matrix_a: sparse.spmatrix, matrix_b: sparse.spmatrix) -> np.ndarray:
    #calculamos coseno fila por fila entre snippet a y snippet b del mismo par
    dot = np.asarray(matrix_a.multiply(matrix_b).sum(axis=1)).ravel()
    norm_a = np.sqrt(np.asarray(matrix_a.multiply(matrix_a).sum(axis=1)).ravel())
    norm_b = np.sqrt(np.asarray(matrix_b.multiply(matrix_b).sum(axis=1)).ravel())
    denom = norm_a * norm_b
    denom[denom == 0.0] = 1e-12
    return dot / denom


def build_pair_features(df: pd.DataFrame, vectorizer: TfidfVectorizer) -> pd.DataFrame:
    #transformamos snippets a espacio tf idf y derivamos features de similitud y tamano
    matrix_a = vectorizer.transform(df["token_text_a"])
    matrix_b = vectorizer.transform(df["token_text_b"])
    cosine_values = rowwise_cosine_similarity(matrix_a, matrix_b)

    char_len_a = df["code_a_clean"].str.len().astype(float)
    char_len_b = df["code_b_clean"].str.len().astype(float)
    line_count_a = df["code_a_clean"].str.count("\n").astype(float) + 1.0
    line_count_b = df["code_b_clean"].str.count("\n").astype(float) + 1.0
    token_count_a = df["tokens_a"].apply(len).astype(float)
    token_count_b = df["tokens_b"].apply(len).astype(float)

    jaccard_values = [
        jaccard_similarity(tokens_a, tokens_b)
        for tokens_a, tokens_b in zip(df["tokens_a"], df["tokens_b"])
    ]
    overlap_values = [
        overlap_ratio(tokens_a, tokens_b)
        for tokens_a, tokens_b in zip(df["tokens_a"], df["tokens_b"])
    ]
    dice_values = [
        dice_similarity(tokens_a, tokens_b)
        for tokens_a, tokens_b in zip(df["tokens_a"], df["tokens_b"])
    ]

    #consolidamos todas las features numéricas en una sola tabla
    features_df = pd.DataFrame(
        {
            "cosine_tfidf": cosine_values,
            "jaccard_tokens": jaccard_values,
            "dice_tokens": dice_values,
            "overlap_unique_tokens": overlap_values,
            "char_len_a": char_len_a,
            "char_len_b": char_len_b,
            "char_len_diff": (char_len_a - char_len_b).abs(),
            "line_count_a": line_count_a,
            "line_count_b": line_count_b,
            "line_count_diff": (line_count_a - line_count_b).abs(),
            "token_count_a": token_count_a,
            "token_count_b": token_count_b,
            "token_count_diff": (token_count_a - token_count_b).abs(),
        },
        index=df.index,
    )
    return features_df


def save_feature_splits(
    split_to_features: dict[str, pd.DataFrame],
    split_to_targets: dict[str, pd.Series],
    output_dir: Path,
    target_name: str,
) -> None:
    #exportamos features por split para inspeccion y trazabilidad experimental
    output_dir.mkdir(parents=True, exist_ok=True)
    for split_name, features_df in split_to_features.items():
        export_df = features_df.copy()
        export_df[target_name] = split_to_targets[split_name].values
        export_df.to_csv(output_dir / f"{split_name}_features.csv", index=False, encoding="utf-8")

# --- baseline.train_baseline ---
#entrenamos modelos simples y guardamos metricas artefactos y predicciones por tarea
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier


def _build_models(
    class_weight,
    seed: int,
) -> dict[str, Any]:
    #definimos un baseline simple con DecisionTreeClassifier
    models = {
        "decision_tree": DecisionTreeClassifier(
            criterion="gini",
            max_depth=12,
            min_samples_leaf=2,
            class_weight=class_weight,
            random_state=seed,
        ),
    }
    return models


def train_and_evaluate_task(
    task_name: str,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    labels: list,
    class_weight,
    output_dir: Path,
    seed: int,
    save_roc_curves: bool = False,
) -> dict[str, Any]:
    #entrenamos y evaluamos ambos modelos sobre val y test
    models = _build_models(
        class_weight=class_weight,
        seed=seed,
    )

    #organizamos carpetas de salida por tipo de artefacto
    models_dir = output_dir / "models"
    metrics_dir = output_dir / "metrics"
    plots_dir = output_dir / "plots"
    preds_dir = output_dir / "predictions"
    models_dir.mkdir(parents=True, exist_ok=True)
    metrics_dir.mkdir(parents=True, exist_ok=True)
    plots_dir.mkdir(parents=True, exist_ok=True)
    preds_dir.mkdir(parents=True, exist_ok=True)

    all_results: list[dict[str, Any]] = []
    best_model_name = None
    best_val_f1 = -1.0

    for model_name, model in models.items():
        #ajustamos el modelo con train y lo persistimos en disco
        model.fit(X_train, y_train)
        joblib.dump(model, models_dir / f"{task_name}_{model_name}.joblib")

        model_metrics: dict[str, Any] = {}
        if hasattr(model, "best_params_"):
            #si usamos GridSearch, guardamos la mejor combinación encontrada
            model_metrics["best_params"] = model.best_params_
        for split_name, (X_split, y_split) in {
            "val": (X_val, y_val),
            "test": (X_test, y_test),
        }.items():
            #predecimos y calculamos metricas por split
            y_pred = model.predict(X_split)
            y_proba = model.predict_proba(X_split) if hasattr(model, "predict_proba") else None
            metrics = evaluate_predictions(y_split, y_pred, labels=labels, y_proba=y_proba)
            model_metrics[split_name] = metrics

            #guardamos matriz de confusion como imagen
            cm_array = np.array(metrics["confusion_matrix"])
            save_confusion_matrix_plot(
                confusion=cm_array,
                labels=[str(lbl) for lbl in labels],
                title=f"{task_name} - {model_name} ({split_name})",
                output_path=plots_dir / f"{task_name}_{model_name}_{split_name}_cm.png",
            )
            if save_roc_curves and y_proba is not None and len(labels) == 2:
                #en binario guardamos la curva ROC para comparar visualmente modelos
                save_roc_curve_plot(
                    y_true=y_split,
                    y_score=np.asarray(y_proba)[:, 1],
                    output_path=plots_dir / f"{task_name}_{model_name}_{split_name}_roc.png",
                    title=f"{task_name} - {model_name} ({split_name}) ROC",
                    pos_label=labels[1],
                )

            #exportamos predicciones para auditoria posterior
            pred_df = pd.DataFrame(
                {
                    "row_index": X_split.index,
                    "y_true": y_split.values,
                    "y_pred": y_pred,
                }
            )
            if y_proba is not None:
                #guardamos probabilidades por clase para facilitar análisis posterior
                proba_array = np.asarray(y_proba)
                for class_idx, class_label in enumerate(labels):
                    pred_df[f"proba_{class_label}"] = proba_array[:, class_idx]
            pred_df.to_csv(
                preds_dir / f"{task_name}_{model_name}_{split_name}_predictions.csv",
                index=False,
                encoding="utf-8",
            )

            all_results.append(
                {
                    "task": task_name,
                    "model_name": model_name,
                    "split": split_name,
                    "metrics": metrics,
                }
            )

        #guardamos resumen de metricas por modelo
        save_json(
            model_metrics,
            metrics_dir / f"{task_name}_{model_name}_metrics.json",
        )

        #elegimos el mejor modelo segun f1 macro en validacion
        val_f1 = float(model_metrics["val"]["f1_macro"])
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_name = model_name

    if best_model_name is None:
        raise RuntimeError(f"No model could be selected for task {task_name}.")

    #armamos resumen final de la tarea y lo guardamos
    summary = {
        "task": task_name,
        "best_model_by_val_f1_macro": best_model_name,
        "best_val_f1_macro": best_val_f1,
        "class_weight": class_weight,
        "results": all_results,
    }
    save_json(summary, metrics_dir / f"{task_name}_summary.json")
    return summary

# --- build_clone_dataset ---
import itertools
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd


@dataclass(frozen=True)
class FileRecord:
    filename: str
    path: Path
    source_group: str
    clone_type: str
    problem_id: int | None


def load_labels(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path, header=None, names=["filename", "label"])
    if df["filename"].duplicated().any():
        dup_count = int(df["filename"].duplicated().sum())
        raise ValueError(f"Duplicate filenames in labels CSV: {dup_count}")
    if not set(df["label"].unique()).issubset({0, 1}):
        raise ValueError("Labels must be binary (0/1).")
    return df


def parse_problem_id(filename: str) -> int | None:
    patterns = [
        r"^Clone_(\d+)\.py$",
        r"^4_Clone_(\d+)\.py$",
        r"^Gpt_false_pair_(\d+)\.py$",
    ]
    for pattern in patterns:
        match = re.match(pattern, filename)
        if match:
            return int(match.group(1))
    return None


def classify_file(path: Path) -> tuple[str, str]:
    path_str = str(path).replace("\\", "/")
    if "/true_semantic_clones/py/MT3/" in path_str:
        return "true_semantic_mt3", "type_III"
    if "/true_semantic_clones/py/T4/" in path_str:
        return "true_semantic_t4", "type_IV"
    if "/false_semantic_clones/py/" in path_str:
        return "false_semantic", "non_clone"
    return "unknown", "unknown"


def discover_python_files(dataset_root: Path) -> dict[str, FileRecord]:
    files = list(dataset_root.glob("true_semantic_clones/py/MT3/*.py"))
    files += list(dataset_root.glob("true_semantic_clones/py/T4/*.py"))
    files += list(dataset_root.glob("false_semantic_clones/py/*.py"))

    mapping: dict[str, FileRecord] = {}
    for path in files:
        source_group, clone_type = classify_file(path)
        mapping[path.name] = FileRecord(
            filename=path.name,
            path=path,
            source_group=source_group,
            clone_type=clone_type,
            problem_id=parse_problem_id(path.name),
        )
    return mapping


def iter_pair_rows(
    file_record: FileRecord,
    snippets: list[str],
    label: int,
    include_code: bool,
    dataset_root: Path,
) -> Iterable[dict]:
    for left_idx, right_idx in itertools.combinations(range(len(snippets)), 2):
        row = {
            "is_clone": int(label),
            "clone_type": file_record.clone_type if label == 1 else "non_clone",
            "source_group": file_record.source_group,
            "filename": file_record.filename,
            "file_path": str(file_record.path.relative_to(dataset_root)),
            "problem_id": file_record.problem_id,
            "snippet_index_a": left_idx,
            "snippet_index_b": right_idx,
        }
        if include_code:
            row["code_a"] = snippets[left_idx]
            row["code_b"] = snippets[right_idx]
        yield row


def build_pair_dataset(
    dataset_root: Path,
    labels_filename: str = "plagiarism_labels.csv",
    include_code: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    labels_path = dataset_root / labels_filename
    labels_df = load_labels(labels_path)
    file_map = discover_python_files(dataset_root)

    missing_files = sorted(set(labels_df["filename"]) - set(file_map))
    if missing_files:
        raise FileNotFoundError(
            f"{len(missing_files)} files listed in labels CSV were not found. "
            f"Examples: {missing_files[:5]}"
        )

    rows: list[dict] = []
    issues: list[dict] = []

    for item in labels_df.itertuples(index=False):
        file_record = file_map[item.filename]
        text = file_record.path.read_text(encoding="utf-8", errors="replace")
        snippets = split_snippets(text)

        if len(snippets) < 2:
            issues.append(
                {
                    "filename": file_record.filename,
                    "reason": "fewer_than_2_snippets",
                    "snippet_count": len(snippets),
                    "source_group": file_record.source_group,
                    "label": int(item.label),
                }
            )
            continue

        if file_record.source_group.startswith("true_semantic") and int(item.label) != 1:
            issues.append(
                {
                    "filename": file_record.filename,
                    "reason": "label_source_mismatch_true_expected_1",
                    "snippet_count": len(snippets),
                    "source_group": file_record.source_group,
                    "label": int(item.label),
                }
            )
        if file_record.source_group == "false_semantic" and int(item.label) != 0:
            issues.append(
                {
                    "filename": file_record.filename,
                    "reason": "label_source_mismatch_false_expected_0",
                    "snippet_count": len(snippets),
                    "source_group": file_record.source_group,
                    "label": int(item.label),
                }
            )

        rows.extend(
            iter_pair_rows(
                file_record=file_record,
                snippets=snippets,
                label=int(item.label),
                include_code=include_code,
                dataset_root=dataset_root,
            )
        )

    dataset_df = pd.DataFrame(rows)
    issues_df = pd.DataFrame(issues)
    return dataset_df, issues_df


# --- run_baseline pipeline ---
def assign_split_column(
    df: pd.DataFrame,
    train_idx: pd.Index,
    val_idx: pd.Index,
    test_idx: pd.Index,
    split_col: str,
) -> pd.DataFrame:
    #etiquetamos cada fila como train val o test a partir de indices ya agrupados
    result = df.copy()
    result[split_col] = "unassigned"
    result.loc[train_idx, split_col] = "train"
    result.loc[val_idx, split_col] = "val"
    result.loc[test_idx, split_col] = "test"
    return result


def _log_split_stats(logger: logging.Logger, title: str, stats: list[dict]) -> None:
    #dejamos en logs una vista clara de filas grupos y clases por split
    logger.info(title)
    for row in stats:
        logger.info(
            "  %s | rows=%s | unique_problem_id=%s | class_dist=%s",
            row["split"],
            row["rows"],
            row["unique_groups"],
            row["class_distribution"],
        )


def _train_task_a(
    prepared_df: pd.DataFrame,
    config: BaselineConfig,
    output_dir: Path,
    logger: logging.Logger,
    balance_train_strategy: str,
) -> dict:
    #entrenamos la tarea binaria clone vs non clone
    split = grouped_train_val_test_split(
        df=prepared_df,
        group_col="problem_id",
        target_col="is_clone",
        seed=config.seed,
        train_size=config.train_size,
        val_size=config.val_size,
        test_size=config.test_size,
    )
    task_df = assign_split_column(
        prepared_df,
        train_idx=split.train_idx,
        val_idx=split.val_idx,
        test_idx=split.test_idx,
        split_col="split_task_a",
    )
    task_df.to_csv(output_dir / "datasets" / "task_a_reconstructed_with_split.csv", index=False)

    #reportamos estadisticas del split para verificar ausencia de leakage
    stats = split_statistics(
        task_df, split_col="split_task_a", target_col="is_clone", group_col="problem_id"
    )
    _log_split_stats(logger, "Task A split statistics (`is_clone`):", stats)
    save_json({"split_stats": stats}, output_dir / "metrics" / "task_a_split_stats.json")

    #separamos dataframes por split
    train_df_raw = task_df[task_df["split_task_a"] == "train"].copy()
    val_df = task_df[task_df["split_task_a"] == "val"].copy()
    test_df = task_df[task_df["split_task_a"] == "test"].copy()

    #balanceamos solo entrenamiento y dejamos val/test intactos para medir generalizacion real
    train_df, balance_info = balance_training_dataframe(
        train_df=train_df_raw,
        target_col="is_clone",
        strategy=balance_train_strategy,
        seed=config.seed,
    )
    save_json(balance_info, output_dir / "metrics" / "task_a_balance_info.json")
    logger.info("Task A balancing: %s", balance_info)

    #aplicamos class_weight balanced cuando el desbalance supera el umbral definido
    ratio = imbalance_ratio(train_df["is_clone"])
    class_weight = "balanced" if ratio >= config.imbalance_threshold else None
    logger.info(
        "Task A class imbalance ratio (train): %.4f | class_weight=%s",
        ratio,
        class_weight,
    )

    #ajustamos tf idf con train y construimos features en train val y test
    vectorizer = fit_tfidf_vectorizer(train_df)
    features_train = build_pair_features(train_df, vectorizer)
    features_val = build_pair_features(val_df, vectorizer)
    features_test = build_pair_features(test_df, vectorizer)

    #exportamos features del baseline para reproducibilidad experimental
    save_feature_splits(
        split_to_features={
            "train": features_train,
            "val": features_val,
            "test": features_test,
        },
        split_to_targets={
            "train": train_df["is_clone"],
            "val": val_df["is_clone"],
            "test": test_df["is_clone"],
        },
        output_dir=output_dir / "features" / "task_a",
        target_name="is_clone",
    )

    #entrenamos y evaluamos el baseline unicamente con decision tree
    summary = train_and_evaluate_task(
        task_name="is_clone",
        X_train=features_train,
        y_train=train_df["is_clone"],
        X_val=features_val,
        y_val=val_df["is_clone"],
        X_test=features_test,
        y_test=test_df["is_clone"],
        labels=[0, 1],
        class_weight=class_weight,
        output_dir=output_dir / "task_a_artifacts",
        seed=config.seed,
        save_roc_curves=True,
    )
    summary["class_imbalance_ratio_train"] = ratio
    summary["class_distribution_train"] = class_distribution(train_df["is_clone"])
    summary["class_distribution_train_before_balance"] = class_distribution(train_df_raw["is_clone"])
    summary["balance_info"] = balance_info
    return summary


def _train_task_b(
    prepared_df: pd.DataFrame,
    config: BaselineConfig,
    output_dir: Path,
    logger: logging.Logger,
    balance_train_strategy: str,
) -> dict:
    #entrenamos la tarea multiclasica solo en positivos para distinguir type_iii y type_iv
    positives_df = prepared_df[prepared_df["is_clone"] == 1].copy()
    split = grouped_train_val_test_split(
        df=positives_df,
        group_col="problem_id",
        target_col="clone_type",
        seed=config.seed + 100,
        train_size=config.train_size,
        val_size=config.val_size,
        test_size=config.test_size,
    )
    task_df = assign_split_column(
        positives_df,
        train_idx=split.train_idx,
        val_idx=split.val_idx,
        test_idx=split.test_idx,
        split_col="split_task_b",
    )
    task_df.to_csv(output_dir / "datasets" / "task_b_positive_with_split.csv", index=False)

    #revisamos que la distribucion por split sea razonable
    stats = split_statistics(
        task_df, split_col="split_task_b", target_col="clone_type", group_col="problem_id"
    )
    _log_split_stats(logger, "Task B split statistics (`clone_type` in positives):", stats)
    save_json({"split_stats": stats}, output_dir / "metrics" / "task_b_split_stats.json")

    #separamos dataframes por split
    train_df_raw = task_df[task_df["split_task_b"] == "train"].copy()
    val_df = task_df[task_df["split_task_b"] == "val"].copy()
    test_df = task_df[task_df["split_task_b"] == "test"].copy()

    #balanceamos solo entrenamiento y dejamos val/test intactos para medir generalizacion real
    train_df, balance_info = balance_training_dataframe(
        train_df=train_df_raw,
        target_col="clone_type",
        strategy=balance_train_strategy,
        seed=config.seed + 100,
    )
    save_json(balance_info, output_dir / "metrics" / "task_b_balance_info.json")
    logger.info("Task B balancing: %s", balance_info)

    #decidimos si usar class_weight balanced para compensar desbalance
    ratio = imbalance_ratio(train_df["clone_type"])
    class_weight = "balanced" if ratio >= config.imbalance_threshold else None
    logger.info(
        "Task B class imbalance ratio (train): %.4f | class_weight=%s",
        ratio,
        class_weight,
    )

    #vectorizamos y creamos features del par para los tres splits
    vectorizer = fit_tfidf_vectorizer(train_df)
    features_train = build_pair_features(train_df, vectorizer)
    features_val = build_pair_features(val_df, vectorizer)
    features_test = build_pair_features(test_df, vectorizer)

    #guardamos features para trazabilidad
    save_feature_splits(
        split_to_features={
            "train": features_train,
            "val": features_val,
            "test": features_test,
        },
        split_to_targets={
            "train": train_df["clone_type"],
            "val": val_df["clone_type"],
            "test": test_df["clone_type"],
        },
        output_dir=output_dir / "features" / "task_b",
        target_name="clone_type",
    )

    #ejecutamos entrenamiento y evaluacion con ambas familias de modelo
    labels = sorted(task_df["clone_type"].unique().tolist())
    summary = train_and_evaluate_task(
        task_name="clone_type",
        X_train=features_train,
        y_train=train_df["clone_type"],
        X_val=features_val,
        y_val=val_df["clone_type"],
        X_test=features_test,
        y_test=test_df["clone_type"],
        labels=labels,
        class_weight=class_weight,
        output_dir=output_dir / "task_b_artifacts",
        seed=config.seed,
    )
    summary["class_imbalance_ratio_train"] = ratio
    summary["class_distribution_train"] = class_distribution(train_df["clone_type"])
    summary["class_distribution_train_before_balance"] = class_distribution(train_df_raw["clone_type"])
    summary["balance_info"] = balance_info
    return summary


def _build_markdown_report(
    output_path: Path,
    config: BaselineConfig,
    metadata_rows: int,
    cleaned_rows: int,
    reconstructed_rows: int,
    dropped_rows: int,
    drop_reasons: dict,
    task_a_summary: dict,
    task_b_summary: dict,
    balance_train_strategy: str,
) -> None:
    #dejamos un reporte breve y legible con datos de reconstruccion y rendimiento
    model_lines = [
        "- Model used: DecisionTreeClassifier.",
    ]
    model_lines.append("- ROC/AUC plots are generated for Task A (`is_clone`).")

    lines = [
        "# Baseline Report",
        "",
        "## Dataset Reconstruction",
        f"- Metadata input rows: **{metadata_rows}**",
        f"- Valid metadata rows after schema cleaning: **{cleaned_rows}**",
        f"- Reconstructed pairs: **{reconstructed_rows}**",
        f"- Dropped rows: **{dropped_rows}**",
        f"- Drop reasons: `{drop_reasons}`",
        "",
        "## Method",
        "- Pair reconstruction from `file_path` + `snippet_index_a/snippet_index_b`.",
        "- Code preprocessing: comment removal + whitespace normalization.",
        "- Tokenization: Python `tokenize` module.",
        "- Pair features: TF-IDF cosine + lexical/length statistics (Jaccard, Dice, overlap).",
        *model_lines,
        "- Grouped split by `problem_id` to prevent leakage.",
        "",
        "## Task A (`is_clone`)",
        f"- Balance strategy (train): **{balance_train_strategy}**",
        f"- Train class distribution before balance: `{task_a_summary.get('class_distribution_train_before_balance', {})}`",
        f"- Train class distribution: `{task_a_summary.get('class_distribution_train', {})}`",
        f"- Imbalance ratio (train): **{task_a_summary.get('class_imbalance_ratio_train', 0):.4f}**",
        f"- Best model by validation F1-macro: **{task_a_summary['best_model_by_val_f1_macro']}**",
        f"- Best validation F1-macro: **{task_a_summary['best_val_f1_macro']:.4f}**",
        "",
        "## Task B (`clone_type` on positives)",
        f"- Balance strategy (train): **{balance_train_strategy}**",
        f"- Train class distribution before balance: `{task_b_summary.get('class_distribution_train_before_balance', {})}`",
        f"- Train class distribution: `{task_b_summary.get('class_distribution_train', {})}`",
        f"- Imbalance ratio (train): **{task_b_summary.get('class_imbalance_ratio_train', 0):.4f}**",
        f"- Best model by validation F1-macro: **{task_b_summary['best_model_by_val_f1_macro']}**",
        f"- Best validation F1-macro: **{task_b_summary['best_val_f1_macro']:.4f}**",
        "",
        "## Reproducibility",
        f"- Random seed: **{config.seed}**",
        f"- Split ratios: train={config.train_size}, val={config.val_size}, test={config.test_size}",
        "",
    ]
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text("\n".join(lines), encoding="utf-8")


def run_pipeline(
    dataset_root: Path,
    metadata_file: str = "clone_pairs_dataset_metadata.csv",
    output_dir: Path | None = None,
    seed: int = 42,
    imbalance_threshold: float = 1.5,
    balance_train_strategy: str = "undersample",
) -> None:
    if output_dir is None:
        output_dir = PROJECT_ROOT / "baseline_outputs"
    config = BaselineConfig(
        dataset_root=dataset_root,
        metadata_filename=metadata_file,
        seed=seed,
        imbalance_threshold=imbalance_threshold,
    )

    #creamos estructura base de carpetas de salida
    output_dir = ensure_dir(output_dir)
    ensure_dir(output_dir / "datasets")
    ensure_dir(output_dir / "metrics")
    ensure_dir(output_dir / "features")
    ensure_dir(output_dir / "reports")

    #preparamos logs y semilla global
    logger = setup_logging(output_dir / "run.log")
    set_global_seed(config.seed)
    logger.info("Starting baseline pipeline.")
    logger.info("Dataset root: %s", config.dataset_root)
    logger.info("Metadata CSV: %s", config.metadata_path)
    logger.info("Model configuration: decision_tree only")
    logger.info("Train balance strategy: %s", balance_train_strategy)

    #cargamos metadata y verificamos su esquema
    metadata_df = load_metadata_csv(config.metadata_path)
    metadata_rows = len(metadata_df)
    logger.info("Loaded metadata rows: %d", metadata_rows)

    schema_ok, schema_errors = validate_metadata_schema(metadata_df)
    if not schema_ok:
        for err in schema_errors:
            logger.error(err)
        raise ValueError("Metadata schema validation failed.")

    #limpiamos metadata y eliminamos filas con valores invalidos
    cleaned_metadata_df, invalid_metadata_df = clean_metadata_rows(metadata_df)
    logger.info(
        "Metadata cleaning: valid_rows=%d | invalid_rows=%d",
        len(cleaned_metadata_df),
        len(invalid_metadata_df),
    )

    #reconstruimos code_a code_b desde los archivos fuente
    reconstructed_df, dropped_df, reconstruction_summary = reconstruct_pairs_from_metadata(
        cleaned_metadata_df, dataset_root=config.dataset_root
    )
    logger.info("Reconstruction summary: %s", reconstruction_summary)

    #unificamos descartes de metadata y descartes de reconstruccion
    if not invalid_metadata_df.empty:
        dropped_df = pd.concat([dropped_df, invalid_metadata_df], ignore_index=True, sort=False)

    #guardamos datasets de trabajo y auditoria
    reconstructed_path = output_dir / "datasets" / "reconstructed_pairs.csv"
    dropped_path = output_dir / "datasets" / "dropped_rows.csv"
    reconstructed_df.to_csv(reconstructed_path, index=False, encoding="utf-8")
    dropped_df.to_csv(dropped_path, index=False, encoding="utf-8")

    logger.info("Saved reconstructed dataset: %s", reconstructed_path)
    logger.info("Saved dropped rows dataset: %s", dropped_path)

    #preprocesamos codigo y tokens antes de extraer features
    prepared_df = prepare_pair_text_fields(reconstructed_df)
    prepared_df.to_csv(
        output_dir / "datasets" / "reconstructed_pairs_preprocessed.csv",
        index=False,
        encoding="utf-8",
    )
    logger.info("Preprocessing completed on %d rows.", len(prepared_df))

    #entrenamos ambas tareas del baseline
    task_a_summary = _train_task_a(
        prepared_df,
        config=config,
        output_dir=output_dir,
        logger=logger,
        balance_train_strategy=balance_train_strategy,
    )
    task_b_summary = _train_task_b(
        prepared_df,
        config=config,
        output_dir=output_dir,
        logger=logger,
        balance_train_strategy=balance_train_strategy,
    )

    #generamos tablas comparativas por tarea y consolidado
    task_a_comp = comparison_table(task_a_summary["results"])
    task_b_comp = comparison_table(task_b_summary["results"])
    task_a_comp.to_csv(output_dir / "metrics" / "task_a_model_comparison.csv", index=False)
    task_b_comp.to_csv(output_dir / "metrics" / "task_b_model_comparison.csv", index=False)
    combined_comp = pd.concat([task_a_comp, task_b_comp], ignore_index=True)
    combined_comp.to_csv(output_dir / "metrics" / "all_tasks_model_comparison.csv", index=False)

    #guardamos resumenes json por tarea
    save_json(task_a_summary, output_dir / "metrics" / "task_a_summary.json")
    save_json(task_b_summary, output_dir / "metrics" / "task_b_summary.json")

    #escribimos reporte final en markdown
    _build_markdown_report(
        output_path=output_dir / "reports" / "baseline_report.md",
        config=config,
        metadata_rows=metadata_rows,
        cleaned_rows=len(cleaned_metadata_df),
        reconstructed_rows=len(reconstructed_df),
        dropped_rows=len(dropped_df),
        drop_reasons=(dropped_df["drop_reason"].value_counts().to_dict() if not dropped_df.empty else {}),
        task_a_summary=task_a_summary,
        task_b_summary=task_b_summary,
        balance_train_strategy=balance_train_strategy,
    )
    logger.info("Baseline report saved to %s", output_dir / "reports" / "baseline_report.md")
    logger.info("Pipeline completed successfully.")




# --- salida al cargar definiciones (notebook) ---
print("=" * 60)
print("[Paso definiciones] Código del pipeline cargado en memoria.")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("plagiarism_labels.csv existe:", (PROJECT_ROOT / "plagiarism_labels.csv").exists())
print("true_semantic_clones/ existe:", (PROJECT_ROOT / "true_semantic_clones").is_dir())
print("false_semantic_clones/ existe:", (PROJECT_ROOT / "false_semantic_clones").is_dir())
print("=" * 60)


## Build pair CSVs

Writes `clone_pairs_dataset.csv`, `clone_pairs_dataset_metadata.csv`, `clone_pairs_issues.csv` under `PROJECT_ROOT`.


In [ ]:
print("\n[Paso dataset pares] Construyendo pares desde etiquetas y archivos .py ...")
METADATA_CSV = PROJECT_ROOT / "clone_pairs_dataset_metadata.csv"
FULL_PAIRS_CSV = PROJECT_ROOT / "clone_pairs_dataset.csv"
ISSUES_CSV = PROJECT_ROOT / "clone_pairs_issues.csv"

full_df, issues_df = build_pair_dataset(
    dataset_root=PROJECT_ROOT,
    include_code=True,
)
print(f"  Filas pares (con código): {len(full_df)}")
print(f"  Filas issues (archivos con <2 snippets o avisos): {len(issues_df)}")

full_df.to_csv(FULL_PAIRS_CSV, index=False, encoding="utf-8")
issues_df.to_csv(ISSUES_CSV, index=False, encoding="utf-8")
meta_df = full_df.drop(columns=["code_a", "code_b"], errors="ignore")
meta_df.to_csv(METADATA_CSV, index=False, encoding="utf-8")

print(f"  Guardado metadata: {METADATA_CSV}")
print(f"  Guardado pares completos: {FULL_PAIRS_CSV}")
print(f"  Guardado issues: {ISSUES_CSV}")
print("\n  is_clone (meta_df):")
print(meta_df["is_clone"].value_counts().sort_index().to_string())
print("\n  clone_type (meta_df):")
print(meta_df["clone_type"].value_counts().to_string())
print("\n  Primeras filas de metadata (sin code_a/code_b):")
try:
    from IPython.display import display
    display(meta_df.head(8))
except Exception:
    print(meta_df.head(8).to_string())
if not issues_df.empty:
    print("\n  Primeras filas de issues:")
    try:
        display(issues_df.head(5))
    except Exception:
        print(issues_df.head(5).to_string())
print("\n[Paso dataset pares] Terminado.\n")


## Class balance (dataset)

Histogram-style view of how many pair rows you have per label in the metadata CSV built above (`is_clone` and `clone_type`).


In [ ]:
print("[Paso gráficos dataset] Distribución de clases en todo el metadata de pares.")
print("is_clone:", dict(meta_df["is_clone"].value_counts()))
print("clone_type:", meta_df["clone_type"].value_counts().to_dict())

import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(
    meta_df["is_clone"].astype(int),
    bins=[-0.5, 0.5, 1.5],
    rwidth=0.85,
    color="#4C72B0",
    edgecolor="white",
)
axes[0].set_xticks([0, 1])
axes[0].set_xlabel("is_clone")
axes[0].set_ylabel("Number of pairs")
axes[0].set_title("Pair rows by is_clone")

vc_type = meta_df["clone_type"].value_counts()
axes[1].bar(
    np.arange(len(vc_type)),
    vc_type.values,
    color=plt.cm.Set2(np.linspace(0, 0.85, len(vc_type))),
    edgecolor="white",
)
axes[1].set_xticks(np.arange(len(vc_type)))
axes[1].set_xticklabels(vc_type.index.astype(str), rotation=30, ha="right")
axes[1].set_ylabel("Number of pairs")
axes[1].set_title("Pair rows by clone_type")

fig.suptitle("Dataset class balance (all splits combined)", fontsize=12, y=1.02)
fig.tight_layout()
plt.show()
print("[Paso gráficos dataset] Gráficos mostrados.\n")


## Run baseline pipeline

Writes `baseline_outputs/`. Optional: change `balance_train_strategy` (`"none"`, `"undersample"`, `"oversample"`).


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "baseline_outputs"

print("\n[Paso baseline] Iniciando pipeline (reconstrucción, features, modelos) ...")
print("  Salida:", OUTPUT_DIR)
print("  Metadata:", PROJECT_ROOT / "clone_pairs_dataset_metadata.csv")
print("  balance_train_strategy: undersample (cambiar en esta celda si quieres)")

run_pipeline(
    dataset_root=PROJECT_ROOT,
    metadata_file="clone_pairs_dataset_metadata.csv",
    output_dir=OUTPUT_DIR,
    seed=42,
    imbalance_threshold=1.5,
    balance_train_strategy="undersample",
)

print("\n[Paso baseline] Pipeline terminado.")
rep = OUTPUT_DIR / "reports" / "baseline_report.md"
print("  Informe:", rep)
for rel in [
    "metrics/task_a_summary.json",
    "metrics/task_b_summary.json",
    "metrics/task_a_balance_info.json",
    "metrics/task_b_balance_info.json",
]:
    p = OUTPUT_DIR / rel
    print(f"  {rel} existe={p.exists()} tamaño_bytes={p.stat().st_size if p.exists() else 0}")
logp = OUTPUT_DIR / "run.log"
if logp.exists():
    lines = logp.read_text(encoding="utf-8").splitlines()
    print("\n  Últimas 12 líneas de run.log:")
    for ln in lines[-12:]:
        print("   ", ln)
print("  (Detalle completo en baseline_outputs/run.log)\n")


## Class balance (training split)

Compares **train only** class counts **before** vs **after** the balancing step inside the baseline (`balance_train_strategy`). Values come from `task_*_balance_info.json` under `baseline_outputs/metrics/`.


In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path


def plot_train_balance(json_path: Path, title: str) -> None:
    data = json.loads(json_path.read_text(encoding="utf-8"))
    print(f"\n  {title}")
    print(f"    archivo: {json_path}")
    print(f"    strategy: {data.get('strategy')}")
    print(f"    filas train antes: {data.get('rows_before')} después: {data.get('rows_after')}")
    print(f"    clases antes: {data['class_distribution_before']}")
    print(f"    clases después: {data['class_distribution_after']}")
    before = data["class_distribution_before"]
    after = data["class_distribution_after"]
    classes = sorted(set(before) | set(after), key=lambda x: str(x))
    x = np.arange(len(classes))
    width = 0.35
    b_vals = [int(before.get(str(c), 0)) for c in classes]
    a_vals = [int(after.get(str(c), 0)) for c in classes]

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(x - width / 2, b_vals, width=width, label="Train before balance", color="#8da0cb")
    ax.bar(x + width / 2, a_vals, width=width, label="Train after balance", color="#fc8d62")
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in classes])
    ax.set_ylabel("Count")
    ax.set_title(f"{title} (strategy={data.get('strategy', '?')})")
    ax.legend()
    fig.tight_layout()
    plt.show()


print("[Paso gráficos entrenamiento] Leyendo balance_info de train (JSON) ...")
metrics_dir = PROJECT_ROOT / "baseline_outputs" / "metrics"
plot_train_balance(metrics_dir / "task_a_balance_info.json", "Task A — is_clone")
plot_train_balance(metrics_dir / "task_b_balance_info.json", "Task B — clone_type (positives only)")
print("\n[Paso gráficos entrenamiento] Terminado.\n")


## View report


In [ ]:
from IPython.display import Markdown, display

report_path = PROJECT_ROOT / "baseline_outputs" / "reports" / "baseline_report.md"
print("[Paso informe] Mostrando Markdown del informe:")
print(" ", report_path)
print("  caracteres:", len(report_path.read_text(encoding="utf-8")))
display(Markdown(report_path.read_text(encoding="utf-8")))
print("\n[Paso informe] Fin del notebook de ejecución típica.\n")
